In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 

# Models
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
# Evaluation
from sklearn.metrics import (
    confusion_matrix, 
    precision_recall_curve,
    auc,
    average_precision_score,
    roc_auc_score,
    f1_score,
    recall_score,
    precision_score,
    classification_report
)

from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
import joblib

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

In [2]:
X_train = joblib.load("artifacts/X_train.pkl")
X_val = joblib.load("artifacts/X_val.pkl")
X_test = joblib.load("artifacts/X_test.pkl")

y_train = joblib.load("artifacts/y_train.pkl")
y_val = joblib.load("artifacts/y_val.pkl")
y_test = joblib.load("artifacts/y_test.pkl")


FileNotFoundError: [Errno 2] No such file or directory: 'artifacts/X_train.pkl'

In [ ]:
#Define Models (with imbalance handling) 
neg, pos = np.bincount(y_train)

scale_pos_weight = neg/pos

models = {
    "LogisticRegression": LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "RandomForest": RandomForestClassifier(class_weight="balanced_subsample", n_estimators=200, random_state=42),
    "SVC": SVC(class_weight="balanced", probability=True, random_state=42), 
    "KNN": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(scale_pos_weight=scale_pos_weight, eval_metric="logloss", random_state=42)
}

tuning goal is:
> Make the model rank customers correctly and output usable probabilities

NOT:

- maximize F1

- minimize cost

- choose final threshold

- Those come later.

# ignore everything above for now

In [ ]:
def evaluate_model_proba(model, X, y):
    probs = model.predict_proba(X)[:, 1] 

    return{
        "ROC_AUC": roc_auc_score(y, probs),
        "PR_AUC": average_precision_score(y, probs)
    }

In [ ]:
# Train & Evaluate on All models 

trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model

rows = []

for name, model in trained_models.items():
    metrics = evaluate_model_proba(model, X_val, y_val)

    rows.append({
        "Model": name,
        "ROC_AUC": metrics["ROC_AUC"],
        "PR_AUC": metrics["PR_AUC"]
    })

leaderboard = pd.DataFrame(rows)
leaderboard = leaderboard.sort_values(by="PR_AUC", ascending=False).reset_index(drop=True)

print(leaderboard)


This is a churn (imbalanced) problem

So:

PR-AUC > ROC-AUC

Why?

ROC-AUC treats false positives as cheap.\
But in churn, contacting wrong customers costs money.

So we rank models by PR-AUC first, ROC-AUC second.

| Rank | Model              | PR-AUC    | ROC-AUC |
| ---- | ------------------ | --------- | ------- |
| 🥇 1 | LogisticRegression | **0.644** | 0.851   |
| 🥈 2 | XGBoost            | 0.634     | 0.822   |
| 🥉 3 | SVC                | 0.608     | 0.822   |
| 4    | RandomForest       | 0.592     | 0.814   |
| 5    | KNN                | 0.510     | 0.771   |
| 6    | DecisionTree       | 0.365     | 0.642   |

The surprising result (and important lesson)

👉 Linear model beat tree ensembles

This actually tells you something about the dataset:

Churn boundary is mostly linear / monotonic\
(not complex interactions)

Very common in telecom / subscription churn datasets.

So boosting models are just overfitting noise.

This is a good sign, not a bad one.

In [ ]:
# Hyperparameter tuning 

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
# Logistic Regression Tuning 

lr = LogisticRegression(max_iter=5000)

param_lr = {
    "C": np.logspace(-3, 3, 60),
    "penalty": ["l2"],
    "solver": ["lbfgs"]
}

lr_search = RandomizedSearchCV(
    lr,
    param_distributions=param_lr,
    n_iter=30,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

lr_search.fit(X_train, y_train)

best_lr = lr_search.best_estimator_
print("Best Logistic params:", lr_search.best_params_)

In [ ]:
xgb = XGBClassifier(
    eval_metric="logloss",
    tree_method="hist",
    random_state=42
)

param_xgb = {
    "n_estimators": [200, 300, 400, 600],
    "max_depth": [3, 4, 5, 6],
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_xgb,
    n_iter=40,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

xgb_search.fit(X_train, y_train)

best_xgb = xgb_search.best_estimator_
print("Best XGB params:", xgb_search.best_params_)


In [ ]:
svc = SVC(probability=True)

param_svc = {
    "C": [0.1, 0.5, 1, 3, 10, 30],
    "gamma": ["scale", 0.01, 0.001, 0.0005],
    "kernel": ["rbf"]
}

svc_search = RandomizedSearchCV(
    svc,
    param_distributions=param_svc,
    n_iter=15,
    scoring="average_precision",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    random_state=42
)

svc_search.fit(X_train, y_train)

best_svc = svc_search.best_estimator_
print("Best SVC params:", svc_search.best_params_)


In [ ]:
tuned_models = {
    "Logistic_Tuned": best_lr,
    "XGB_Tuned": best_xgb,
    "SVC_Tuned": best_svc
}

rows = []
for name, model in tuned_models.items():
    probs = model.predict_proba(X_val)[:,1]
    rows.append({
        "Model": name,
        "ROC_AUC": roc_auc_score(y_val, probs),
        "PR_AUC": average_precision_score(y_val, probs)
    })

tuned_leaderboard = pd.DataFrame(rows).sort_values(by="PR_AUC", ascending=False)
print(tuned_leaderboard)


In [ ]:
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier(
    estimators=[
        ("lr", best_lr),
        ("xgb", best_xgb),
        ("svc", best_svc)
    ],
    voting="soft",
    weights=[1, 2, 1]  # give XGB more importance (best PR-AUC)
)

ensemble.fit(X_train, y_train)


In [ ]:
def recall_at_k(y_true, prob, k = 0.2):
    cutoff = int(len(probs)* k)
    idx = np.argsort(prob)[::-1][:cutoff]
    return y_true.iloc[idx].sum()/y_true.sum()

In [ ]:
probs = ensemble.predict_proba(X_val)[:,1]

print("Recall@10%:", recall_at_k(y_val, probs, 0.10))
print("Recall@20%:", recall_at_k(y_val, probs, 0.20))
print("Recall@30%:", recall_at_k(y_val, probs, 0.30))


In [ ]:
# Calibrate using validation set 
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(
    estimator=ensemble,
    method="isotonic", 
    cv = "prefit"
)

calibrated_model.fit(X_val, y_val)

In [ ]:
from sklearn.metrics import brier_score_loss

probs_before = ensemble.predict_proba(X_val)[:,1]
probs_after  = calibrated_model.predict_proba(X_val)[:,1]

print("Brier before:", brier_score_loss(y_val, probs_before))
print("Brier after :", brier_score_loss(y_val, probs_after))


In [ ]:
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

prob_true_before, prob_pred_before = calibration_curve(y_val, probs_before, n_bins=10)
prob_true_after, prob_pred_after = calibration_curve(y_val, probs_after, n_bins=10)

plt.figure(figsize=(6,6))
plt.plot(prob_pred_before, prob_true_before, marker='o', label="Before")
plt.plot(prob_pred_after, prob_true_after, marker='o', label="After")
plt.plot([0,1],[0,1],'--',color='gray')

plt.xlabel("Predicted Probability")
plt.ylabel("Actual Churn Rate")
plt.title("Calibration Curve")
plt.legend()
plt.show()


In [ ]:
# Ensemble
probs_ens = ensemble.predict_proba(X_val)[:,1]

print("Ensemble ROC-AUC:", roc_auc_score(y_val, probs_ens))
print("Ensemble PR-AUC:", average_precision_score(y_val, probs_ens))

# Calibrated
probs_cal = calibrated_model.predict_proba(X_val)[:,1]

print("Calibrated ROC-AUC:", roc_auc_score(y_val, probs_cal))
print("Calibrated PR-AUC:", average_precision_score(y_val, probs_cal))


In [ ]:
probs = calibrated_model.predict_proba(X_val)[:,1]

print("Recall@10%:", recall_at_k(y_val, probs, 0.10))
print("Recall@20%:", recall_at_k(y_val, probs, 0.20))
print("Recall@30%:", recall_at_k(y_val, probs, 0.30))


### Final test on Test set

In [ ]:
# Ensemble
probs_ens = ensemble.predict_proba(X_test)[:,1]

print("Ensemble ROC-AUC:", roc_auc_score(y_test, probs_ens))
print("Ensemble PR-AUC:", average_precision_score(y_test, probs_ens))

# Calibrated
probs_cal = calibrated_model.predict_proba(X_test)[:,1]

print("Calibrated ROC-AUC:", roc_auc_score(y_test, probs_cal))
print("Calibrated PR-AUC:", average_precision_score(y_test, probs_cal))


# Recall 

probs = calibrated_model.predict_proba(X_test)[:,1]

print("Recall@10%:", recall_at_k(y_test, probs, 0.10))
print("Recall@20%:", recall_at_k(y_test, probs, 0.20))
print("Recall@30%:", recall_at_k(y_test, probs, 0.30))

In [ ]:
# Save the trained model 
joblib.dump(calibrated_model, "artifacts/churn_model_calibrated.pkl")

In [ ]:
import json

model_info = {
    "model_type": "SoftVoting + IsotonicCalibration",
    "target": "Churn Probability",
    "features_version": "v1",
    "notes": "Threshold decided in decision notebook"
}

with open("model_info.json", "w") as f:
    json.dump(model_info, f, indent=4)


In [ ]:
import joblib

model = joblib.load("artifacts/churn_model_calibrated.pkl")
preprocessor = joblib.load("artifacts/preprocessor.pkl")

probs = model.predict_proba(X_test)[:,1]
print(probs[:5])


In [ ]:
import shap

feature_names = preprocessor.get_feature_names_out()

X_val_df = pd.DataFrame(X_val, columns=feature_names).reset_index(drop=True)
y_val_series = pd.Series(y_val).reset_index(drop=True)



sample_size = 300
half = sample_size // 2

pos = X_val_df[y_val_series == 1].sample(half, random_state=42)
neg = X_val_df[y_val_series == 0].sample(half, random_state=42)

X_shap = pd.concat([pos, neg])

# background sample (important for speed + stability)
X_train_df = pd.DataFrame(X_train, columns=feature_names)
background = shap.sample(X_train_df, 50, random_state=42)


def predict_fn(X):
    return calibrated_model.predict_proba(X)[:, 1]

explainer = shap.Explainer(
    calibrated_model.predict_proba,
    background
)

explainer = shap.KernelExplainer(predict_fn, background)

shap_values = explainer.shap_values(X_shap, nsamples=200)


In [ ]:
shap.summary_plot(shap_values, X_shap)


In [ ]:
shap.summary_plot(shap_values, X_shap, plot_type="bar", max_display=10)


In [ ]:
shap_exp = shap.Explanation(
    values=shap_values,
    base_values=explainer.expected_value,
    data=X_shap,
    feature_names=X_shap.columns
) 

shap.plots.beeswarm(shap_exp, max_display=10)


In [ ]:
# mean absolute SHAP value = global importance
importance = np.abs(shap_exp.values).mean(axis=0)

feature_importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance
}).sort_values("Importance", ascending=False).head(10)


plt.figure(figsize=(10,8))
sns.barplot(
    data=feature_importance,
    y="Feature",
    x="Importance",
    palette="viridis"
)

plt.title("Top 10 Most Important Features (Final Ensemble Model)", fontsize=14, fontweight='bold')
plt.xlabel("Mean |SHAP impact on churn probability|")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.to_string(index=False))

In [ ]:
shap.plots.waterfall(shap_exp[55])

In [ ]:
shap.force_plot(
    explainer.expected_value,
    shap_values[1],
    X_shap.iloc[0],
    matplotlib=True
)


In [ ]:
# final model with preprocessor and churn prediction 

# load your existing files
preprocessor = joblib.load("artifacts/preprocessor.pkl")
model = joblib.load("artifacts/churn_model_calibrated.pkl")

# combine them
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

# save final production model
joblib.dump(pipeline, "churn_pipeline.pkl")

print("Final pipeline saved!")

In [ ]:
pipeline.feature_names_in_